# 06 — Gold: FactOrderFulfillment (Accumulating Snapshot)

Grain: uma linha por `OrderID`.

**Padrão Kimball — Accumulating Snapshot:** cada pedido tem milestones que se completam ao longo do tempo:
1. `OrderDate` — sempre preenchida
2. `RequiredDate` — sempre preenchida
3. `ShippedDate` — NULL até o envio; quando chega, a linha é **ATUALIZADA**

**Técnica DuckDB (sem MERGE):**
- `UPDATE ... FROM` para atualizar pedidos quando ShippedDate chega
- `INSERT INTO ... WHERE NOT EXISTS` para inserir novos pedidos

Este é o padrão mais sofisticado — demonstra UPDATE em fato dimensional.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# Accumulating Snapshot — lógica em 2 passos:
# PASSO 1: UPDATE pedidos existentes onde ShippedDate chegou/mudou
# PASSO 2: INSERT novos pedidos (ainda não estão no fact)
# ============================================================
# Nota: DuckDB 0.10 tem bug com UPDATE...RETURNING em tabelas com UNIQUE constraint.
# Alternativa: contar com SELECT COUNT antes do UPDATE.

# Contar pedidos a atualizar no PASSO 1
n_to_update = conn.execute("""
    SELECT COUNT(*)
    FROM gold.FactOrderFulfillment f
    JOIN bronze.orders o ON f.OrderID = o.OrderID
    WHERE o.ShippedDate IS NOT NULL
      AND (
          f.ShippedDateKey IS NULL
          OR f.ShippedDateKey <> CAST(strftime(o.ShippedDate::DATE, '%Y%m%d') AS INTEGER)
      )
""").fetchone()[0]

# PASSO 1: UPDATE quando ShippedDate chega ou muda
conn.execute("""
    UPDATE gold.FactOrderFulfillment f
    SET
        ShippedDateKey = CAST(strftime(o.ShippedDate::DATE, '%Y%m%d') AS INTEGER),
        DaysToShip     = DATEDIFF('day', o.OrderDate::DATE, o.ShippedDate::DATE),
        IsLate         = o.ShippedDate::DATE > o.RequiredDate::DATE,
        LoadTimestamp  = current_timestamp
    FROM bronze.orders o
    WHERE f.OrderID = o.OrderID
      AND o.ShippedDate IS NOT NULL
      AND (
          f.ShippedDateKey IS NULL
          OR f.ShippedDateKey <> CAST(strftime(o.ShippedDate::DATE, '%Y%m%d') AS INTEGER)
      )
""")
print(f"PASSO 1 — UPDATE: {n_to_update} pedidos atualizados (ShippedDate chegou/mudou)")

# PASSO 2: INSERT novos pedidos (não existem em gold.FactOrderFulfillment)
conn.execute("""
    INSERT INTO gold.FactOrderFulfillment
    SELECT
        CAST(hash(CAST(o.OrderID AS VARCHAR)) % 2147483647 AS INTEGER) AS FulfillmentSK,
        o.OrderID,
        dc.CustomerSK,
        de.EmployeeSK,
        ds.ShipperSK,
        CAST(strftime(o.OrderDate::DATE,    '%Y%m%d') AS INTEGER) AS OrderDateKey,
        CAST(strftime(o.RequiredDate::DATE, '%Y%m%d') AS INTEGER) AS RequiredDateKey,
        CASE WHEN o.ShippedDate IS NOT NULL
             THEN CAST(strftime(o.ShippedDate::DATE, '%Y%m%d') AS INTEGER)
             ELSE NULL END                                         AS ShippedDateKey,
        o.Freight,
        o.ShipCountry,
        CASE WHEN o.ShippedDate IS NOT NULL
             THEN DATEDIFF('day', o.OrderDate::DATE, o.ShippedDate::DATE)
             ELSE NULL END                                         AS DaysToShip,
        CASE WHEN o.ShippedDate IS NOT NULL
             THEN o.ShippedDate::DATE > o.RequiredDate::DATE
             ELSE NULL END                                         AS IsLate,
        current_timestamp                                          AS LoadTimestamp
    FROM bronze.orders o
    -- DimCustomer: versão vigente na data do pedido
    JOIN gold.DimCustomer dc
        ON o.CustomerID = dc.CustomerID
        AND o.OrderDate::DATE >= dc.ValidFrom
        AND o.OrderDate::DATE <  dc.ValidTo
    JOIN gold.DimEmployee de ON o.EmployeeID = de.EmployeeID
    LEFT JOIN gold.DimShipper ds ON o.ShipVia = ds.ShipperID
    WHERE NOT EXISTS (
        SELECT 1 FROM gold.FactOrderFulfillment f WHERE f.OrderID = o.OrderID
    )
""")

n = conn.execute("SELECT COUNT(*) FROM gold.FactOrderFulfillment").fetchone()[0]
print(f"PASSO 2 — INSERT: novos pedidos inseridos")
print(f"\nFactOrderFulfillment total: {n} linhas (esperado: 830)")

# Status
shipped = conn.execute("SELECT COUNT(*) FROM gold.FactOrderFulfillment WHERE ShippedDateKey IS NOT NULL").fetchone()[0]
pending = conn.execute("SELECT COUNT(*) FROM gold.FactOrderFulfillment WHERE ShippedDateKey IS NULL").fetchone()[0]
print(f"Enviados: {shipped} | Pendentes: {pending}")

PASSO 1 — UPDATE: 0 pedidos atualizados (ShippedDate chegou/mudou)
PASSO 2 — INSERT: novos pedidos inseridos

FactOrderFulfillment total: 830 linhas (esperado: 830)
Enviados: 809 | Pendentes: 21


In [3]:
# ============================================================
# LAB: Simular envio de pedido pendente
# ============================================================
pending_order = conn.execute("""
    SELECT OrderID FROM gold.FactOrderFulfillment
    WHERE ShippedDateKey IS NULL LIMIT 1
""").fetchone()

if pending_order:
    oid = pending_order[0]
    print(f"Pedido pendente escolhido: OrderID={oid}")
    print("\nEstado antes:")
    print(conn.execute(f"""
        SELECT OrderID, OrderDateKey, RequiredDateKey, ShippedDateKey, DaysToShip, IsLate
        FROM gold.FactOrderFulfillment WHERE OrderID = {oid}
    """).fetchdf().to_string(index=False))

    # Simular: atualizar ShippedDate na bronze
    conn.execute(f"""
        UPDATE bronze.orders
        SET ShippedDate = TIMESTAMP '2026-03-08 00:00:00'
        WHERE OrderID = {oid}
    """)
    print(f"\nbronze.orders: ShippedDate de OrderID={oid} atualizado para 2026-03-08.")

    # Contar pendentes antes do re-processamento
    n_pending_before = conn.execute(
        "SELECT COUNT(*) FROM gold.FactOrderFulfillment WHERE ShippedDateKey IS NULL"
    ).fetchone()[0]

    # Re-executar PASSO 1 (UPDATE) — sem RETURNING (bug DuckDB 0.10)
    conn.execute("""
        UPDATE gold.FactOrderFulfillment f
        SET
            ShippedDateKey = CAST(strftime(o.ShippedDate::DATE, '%Y%m%d') AS INTEGER),
            DaysToShip     = DATEDIFF('day', o.OrderDate::DATE, o.ShippedDate::DATE),
            IsLate         = o.ShippedDate::DATE > o.RequiredDate::DATE,
            LoadTimestamp  = current_timestamp
        FROM bronze.orders o
        WHERE f.OrderID = o.OrderID
          AND o.ShippedDate IS NOT NULL
          AND f.ShippedDateKey IS NULL
    """)

    n_pending_after = conn.execute(
        "SELECT COUNT(*) FROM gold.FactOrderFulfillment WHERE ShippedDateKey IS NULL"
    ).fetchone()[0]
    print(f"Re-executado: {n_pending_before - n_pending_after} pedidos atualizados")

    print("\nEstado depois (ShippedDateKey, DaysToShip, IsLate agora preenchidos):")
    print(conn.execute(f"""
        SELECT OrderID, OrderDateKey, RequiredDateKey, ShippedDateKey, DaysToShip, IsLate
        FROM gold.FactOrderFulfillment WHERE OrderID = {oid}
    """).fetchdf().to_string(index=False))
else:
    print("Todos os pedidos já foram enviados.")

Pedido pendente escolhido: OrderID=11008

Estado antes:
 OrderID  OrderDateKey  RequiredDateKey ShippedDateKey DaysToShip IsLate
   11008      19980408         19980506           None       None   None

bronze.orders: ShippedDate de OrderID=11008 atualizado para 2026-03-08.
Re-executado: 1 pedidos atualizados

Estado depois (ShippedDateKey, DaysToShip, IsLate agora preenchidos):
 OrderID  OrderDateKey  RequiredDateKey  ShippedDateKey  DaysToShip  IsLate
   11008      19980408         19980506        20260308       10196    True


In [4]:
# ============================================================
# Validações
# ============================================================
print("1. Contagem total (esperado: 830):")
print(f"   {conn.execute('SELECT COUNT(*) FROM gold.FactOrderFulfillment').fetchone()[0]}")

print("\n2. Grain: 1 linha por OrderID")
dups = conn.execute("""
    SELECT COUNT(*) FROM (
        SELECT OrderID FROM gold.FactOrderFulfillment
        GROUP BY OrderID HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print(f"   Pedidos com mais de 1 linha: {dups} (esperado: 0)")

print("\n3. Enviados vs pendentes:")
print(conn.execute("""
    SELECT
        ShippedDateKey IS NULL AS IsPending,
        COUNT(*) AS Qtd
    FROM gold.FactOrderFulfillment
    GROUP BY IsPending
""").fetchdf().to_string(index=False))

print("\n4. IsLate — distribuição:")
print(conn.execute("""
    SELECT IsLate, COUNT(*) AS Qtd
    FROM gold.FactOrderFulfillment
    WHERE ShippedDateKey IS NOT NULL
    GROUP BY IsLate
""").fetchdf().to_string(index=False))

print("\n5. DaysToShip — estatísticas:")
print(conn.execute("""
    SELECT
        ROUND(AVG(DaysToShip), 1) AS avg_days,
        MIN(DaysToShip) AS min_days,
        MAX(DaysToShip) AS max_days
    FROM gold.FactOrderFulfillment
    WHERE DaysToShip IS NOT NULL
""").fetchdf().to_string(index=False))

conn.close()

1. Contagem total (esperado: 830):
   830

2. Grain: 1 linha por OrderID
   Pedidos com mais de 1 linha: 0 (esperado: 0)

3. Enviados vs pendentes:
 IsPending  Qtd
     False  810
      True   20

4. IsLate — distribuição:
 IsLate  Qtd
  False  772
   True   38

5. DaysToShip — estatísticas:
 avg_days  min_days  max_days
     21.1         1     10196
